# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print("\nKeywords:", getattr(meta, 'keywords', None))
print("Published on:", getattr(meta, 'datePublished', None))
print("Citation:", getattr(meta, 'citeAs', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all `@id`s of record sets. For each record set, we enumerate all fields and their associated columns, referencing everything by `@id` as defined in the Croissant schema.

In [ ]:
record_set_ids = [rs['@id'] for rs in dataset._data.get('recordSet', [])]

print("Available Record Sets (by @id):")
for i, rec in enumerate(dataset._data.get('recordSet', [])):
    print(f"{i+1}. {rec.get('@id')}")

field_map = {}

# For each record set, list fields and columns by @id
for rec in dataset._data.get('recordSet', []):
    rid = rec.get('@id')
    print(f"\nRecord Set: {rid}")
    fields = rec.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_map[rid] = []
    for field in fields:
        fid = field.get('@id')
        field_map[rid].append(fid)
        print(f"  Field: {fid}")
        columns = field.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            cid = col.get('@id')
            print(f"    Column: {cid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# If no record sets are defined, this dataset may attach its main table via distributions or file objects.
if not record_set_ids:
    print("No explicit record sets found in schema. Attempting to extract record set IDs by standard Croissant conventions...")
    # Try to infer (some Croissant datasets use a file as a record set)
    record_set_ids = []
    # Check if there is a CSV or main table file in schema
    if 'distribution' in dataset._data:
        for dist in dataset._data['distribution']:
            record_set_ids.append(dist['@id'])

print('Record sets for extraction:', record_set_ids)
dataframes = {}

for record_set in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records from record set {record_set}")
    except Exception as e:
        print(f"Could not load records from {record_set}: {e}")

if dataframes:
    sample_rs = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set {sample_rs}:")
    print(dataframes[sample_rs].columns.tolist())
    display(dataframes[sample_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This example will filter based on a numeric column (e.g., 'Coverage(%)') if present, normalize it, and group by a categorical attribute such as 'Gene Symbol' or similar. Please adjust field IDs/names below as needed depending on the table structure you see above.

In [ ]:
## This block demonstrates EDA for the loaded DataFrame, referencing fields by @id where possible.

# Update these to use the appropriate field/column names or @ids one finds in previous outputs
df = dataframes[sample_rs]

# Guess column - look for one containing 'Coverage' and another categorical for grouping
numeric_field_candidates = [col for col in df.columns if 'coverage' in col.lower() or '%' in col or 'abundance' in col.lower() or 'MW' in col]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0]

# Attempt to select a grouping field
group_field_candidates = [col for col in df.columns if 'gene' in col.lower() or 'description' in col.lower() or 'protein' in col.lower()]
group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]

# Coerce to numeric just in case
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} rows")
print(filtered_df[[numeric_field, group_field]].head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df[[numeric_field]].head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If group_field is categorical and not too many categories, plot mean value by group
if group_field in df.columns and df[group_field].nunique() <= 20:
    means_by_group = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=means_by_group.index, y=means_by_group.values)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded mass spectrometry summary data from the FAIR^2 Croissant schema.
- Key record sets and fields were referenced by their `@id` as per best practices.
- Data was filtered, normalized, and grouped by selected protein- or gene-level attributes.
- Data distributions and group-wise summaries were visualized.
  
You can further extend this notebook by applying downstream machine learning, deeper statistics, or domain-specific analyses as appropriate.